In [2]:
import pandas as pd
import numpy as np
import os
import networkx as nx

BASE_PATH = ".."
PROCESSED_PATH = os.path.join(BASE_PATH, "data", "processed")

TRANS_PATH = os.path.join(
    PROCESSED_PATH,
    "transactions.csv"
)

trans_df = pd.read_csv(TRANS_PATH)

# Create account IDs
trans_df["Sender Account ID"] = (
    trans_df["From Bank"].astype(str)
    + "_"
    + trans_df["Account"].astype(str)
)

trans_df["Receiver Account ID"] = (
    trans_df["To Bank"].astype(str)
    + "_"
    + trans_df["Account.1"].astype(str)
)

print("Transactions:", trans_df.shape)

Transactions: (6924049, 14)


In [3]:
# Keep only the information required to construct the graph
edges = trans_df[
    ["Sender Account ID", "Receiver Account ID"]
].copy()

# Remove invalid/self relationships
edges = edges.dropna()

# Create unique account-to-account relationships
edges = edges.drop_duplicates()

print("Unique account relationships:", len(edges))

# Directed graph
G = nx.DiGraph()

G.add_edges_from(
    edges.itertuples(index=False, name=None)
)

print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())

Unique account relationships: 1384862
Number of nodes: 705907
Number of edges: 1384862


In [5]:
# Directed degree features
in_degree = dict(G.in_degree())
out_degree = dict(G.out_degree())

# PageRank
pagerank = nx.pagerank(
    G,
    alpha=0.85,
    max_iter=100
)

print("Degree and PageRank calculated.")

Degree and PageRank calculated.


In [6]:
# Convert to undirected graph for structural analysis
G_undirected = G.to_undirected()

# Triangle count for every account
triangle_count = nx.triangles(G_undirected)

# Community detection
communities = nx.community.louvain_communities(
    G_undirected,
    seed=42
)

# Map each account to its community ID and size
community_id = {}
community_size = {}

for cid, community in enumerate(communities):
    size = len(community)

    for account in community:
        community_id[account] = cid
        community_size[account] = size

print("Communities:", len(communities))
print("Triangle counts calculated.")

Communities: 162495
Triangle counts calculated.


In [7]:
# Helper function
def get_graph_features(account_id):
    return {
        "In Degree": in_degree.get(account_id, 0),
        "Out Degree": out_degree.get(account_id, 0),
        "PageRank": pagerank.get(account_id, 0),
        "Triangle Count": triangle_count.get(account_id, 0),
        "Community ID": community_id.get(account_id, -1),
        "Community Size": community_size.get(account_id, 0)
    }


# Sender features
sender_features = pd.DataFrame(
    trans_df["Sender Account ID"]
    .map(get_graph_features)
    .tolist()
)

sender_features.columns = [
    "Sender In Degree",
    "Sender Out Degree",
    "Sender PageRank",
    "Sender Triangle Count",
    "Sender Community ID",
    "Sender Community Size"
]

# Receiver features
receiver_features = pd.DataFrame(
    trans_df["Receiver Account ID"]
    .map(get_graph_features)
    .tolist()
)

receiver_features.columns = [
    "Receiver In Degree",
    "Receiver Out Degree",
    "Receiver PageRank",
    "Receiver Triangle Count",
    "Receiver Community ID",
    "Receiver Community Size"
]

# Final dataset
graph_features_df = pd.concat(
    [
        trans_df[
            [
                "Transaction ID",
                "Sender Account ID",
                "Receiver Account ID"
            ]
        ].reset_index(drop=True),
        sender_features.reset_index(drop=True),
        receiver_features.reset_index(drop=True)
    ],
    axis=1
)

print("Shape:", graph_features_df.shape)
print(graph_features_df.head())

# Save
OUTPUT_PATH = os.path.join(
    PROCESSED_PATH,
    "graph_features.csv"
)

graph_features_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nSaved:", OUTPUT_PATH)

Shape: (6924049, 15)
   Transaction ID Sender Account ID Receiver Account ID  Sender In Degree  \
0               1      11_8000ECA90        11_8000ECA90                33   
1               2    3402_80021DAD0      3402_80021DAD0                 1   
2               3      11_8000ECA90      1120_8006AA910                33   
3               4    3814_8006AD080      3814_8006AD080                 1   
4               5      20_8006AD530        20_8006AD530                23   

   Sender Out Degree  Sender PageRank  Sender Triangle Count  \
0                  7         0.000005                      0   
1                  1         0.000001                      0   
2                  7         0.000005                      0   
3                  1         0.000001                      0   
4                  3         0.000004                      0   

   Sender Community ID  Sender Community Size  Receiver In Degree  \
0                   14                  79083                 